In [1]:
# env: nnunetv2 + nibabel for the FLAIR-removal counterfactual (CPU session, no GPU quota used)
import subprocess, sys
try:
    import nnunetv2
    print('nnunetv2 present')
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nnunetv2'], check=True)
    print('nnunetv2 installed')
import nibabel, torch
print('nibabel', nibabel.__version__, '| torch', torch.__version__, '| cuda:', torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 3.5 MB/s eta 0:00:00
nnunetv2 installed
nibabel 5.4.2 | torch 2.11.0+cpu | cuda: False


In [2]:
# stage manifest (Model A split) + per-case modality files for the 56 test cases
import os, glob, json, subprocess, pandas as pd
os.chdir('/kaggle/working')
if not os.path.isdir('Neuroshield'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pankurajasaivardhan/Neuroshield.git'], check=True)
DS = '/kaggle/input/datasets/rajasaivardhan/neuroshield-track2-model-a'
splits = json.load(open(glob.glob(os.path.join(DS, '**/splits_final.json'), recursive=True)[0]))
fold = splits[0]
test_ids = sorted({os.path.basename(p).replace('.nii.gz', '').replace('.nii', '')
                   for p in glob.glob(os.path.join(DS, '**/test_predictions/**/*.nii*'), recursive=True)})
print('fold0 train/val:', len(fold['train']), len(fold['val']), '| test:', len(test_ids))
rows = [(c, 'train') for c in fold['train']] + [(c, 'val') for c in fold['val']] + [(c, 'test') for c in test_ids]
pd.DataFrame(rows, columns=['case_id', 'split']).to_csv('/kaggle/working/manifest.csv', index=False)
GT = glob.glob('/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]
MODS = ['flair', 't1', 't1ce', 't2']
n = 0
for cid in test_ids:
    out = os.path.join('/kaggle/working/brats', cid)
    os.makedirs(out, exist_ok=True)
    for m in MODS + ['seg']:
        src = os.path.join(GT, cid, f'{cid}_{m}.nii')
        dst = os.path.join(out, f'{cid}_{m}.nii.gz')
        if not os.path.exists(dst):
            os.symlink(os.path.realpath(src), dst)
    n += 1
print('staged test cases:', n)

Cloning into 'Neuroshield'...


fold0 train/val: 249 63 | test: 56
staged test cases: 56


In [3]:
# FLAIR-removal counterfactual on the held-out test split (2 nnU-Net predict passes on CPU)
import subprocess, os, glob
os.chdir('/kaggle/working/Neuroshield')
cands = [p for p in glob.glob('/kaggle/input/datasets/rajasaivardhan/neuroshield-track2-model-a/**/model', recursive=True)
         if os.path.isdir(os.path.join(p, 'Dataset001_BraTS2020'))]
ART = cands[0]
print('artifact:', ART)
r = subprocess.run(['python', '-m', 'track2_brain_tumor.src.modality_removal',
                    '--manifest', '/kaggle/working/manifest.csv',
                    '--raw-data', '/kaggle/working/brats',
                    '--artifact', ART,
                    '--remove', 'flair', '--split', 'test',
                    '--out', '/kaggle/working/outputs', '--config', '2d', '--fold', '0'])
print('exit', r.returncode)
assert r.returncode == 0, 'modality_removal failed - see log'

artifact: /kaggle/input/datasets/rajasaivardhan/neuroshield-track2-model-a/neuroshield_track2_model_a/model

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

perform_everything_on_device=True is only supported for cuda devices! Setting this to False
There are 56 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 56 cases that I would like to predict

Predicting BraTS20_Training_004:
perform_everything_on_device: False


100%|██████████| 140/140 [00:21<00:00,  6.44it/s]


sending off prediction to background worker for resampling and export
done with BraTS20_Training_004

Predicting BraTS20_Training_013:
perform_everything_on_device: False


100%|██████████| 140/140 [00:21<00:00,  6.47it/s]


sending off prediction to background worker for resampling and export
done with BraTS20_Training_013

Predicting BraTS20_Training_014:
perform_everything_on_device: False


...

  0%|          | 0/142 [00:00<?, ?it/s]

sending off prediction to background worker for resampling and export
done with BraTS20_Training_360

Predicting BraTS20_Training_362:
perform_everything_on_device: False


100%|██████████| 142/142 [00:23<00:00,  6.07it/s]


sending off prediction to background worker for resampling and export
done with BraTS20_Training_362
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.
case 1/56 BraTS20_Training_004: wt_dice=0.0905
case 2/56 BraTS20_Training_013: wt_dice=0.0013
case 3/56 BraTS20_Training_014: wt_dice=0.4469
case 4/56 BraTS20_Training_016: wt_dice=0.0768
case 5/56 BraTS20_Training_017: wt_dice=0.2671
case 6/56 BraTS20_Training_023: wt_dice=0.4341
case 7/56 BraTS20_Training_041: wt_dice=0.1429
case 8/56 BraTS20_Training_045: wt_dice=0.4168
case 9/56 BraTS20_Training_048: wt_dice=0.0077
case 10/56 BraTS20_Training_050: wt_dice=0.5803
case 11/56 BraTS20_Training_053: wt_dice=0.2480
case 12/56 BraTS20_Training_058: wt_dice=0.0171
case 13/56 BraTS20_Training_064: wt_dice=0.6145
case 14/56 BraTS20_Training_072: wt_dice=0.4489
case 15/56 BraTS20_Training_080: wt_dice=0.1401
case 16/56 BraTS20_Training_082: wt_dice=0.3347
case 17/56 BraTS20_Training_102: wt_dice=0.1410
case 18/56 BraTS20_Training_111: wt_dice=0.6504
case 19/56 BraTS20_Training_112: wt_dice=0.5439
case 20/56 BraTS20_Training_113: wt_dice=0.5306
case 21/56 BraTS20_